# Advanced Topics in LangGraph\n\n## Tutorials Covered:\n1. Complex Decision Trees\n2. Recursive Reasoning\n3. Multi-Modal Graphs\n4. Performance Optimization

In [1]:
# Install required packages for advanced topics
%pip install -q langgraph langchain-openai python-dotenv

In [2]:
# Import necessary libraries for advanced topics
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List, Dict, Any
import operator
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langchain.schema import AIMessage, HumanMessage, SystemMessage

# Load environment variables
load_dotenv()

# Initialize LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

## 1. Complex Decision Trees\n\nLearning objectives:\n- Build sophisticated branching logic\n- Implement complex conditional flows\n- Design advanced decision-making structures

In [3]:
# Tutorial 1: Complex Decision Trees

# Define state structure
class DecisionTreeState(TypedDict):
    user_input: str
    decision_path: List[str]
    current_decision: str
    decision_result: str
    context: Dict[str, Any]

# Define decision nodes
def classify_input(state: DecisionTreeState):
    user_input = state['user_input']
    
    # Simple classifier based on keywords
    if 'urgent' in user_input.lower() or 'emergency' in user_input.lower():
        decision = 'URGENT_CASE'
    elif 'technical' in user_input.lower() or 'bug' in user_input.lower() or 'error' in user_input.lower():
        decision = 'TECHNICAL_ISSUE'
    elif 'billing' in user_input.lower() or 'payment' in user_input.lower() or 'invoice' in user_input.lower():
        decision = 'BILLING_QUESTION'
    else:
        decision = 'GENERAL_INQUIRY'
    
    # Update decision path
    new_path = state.get('decision_path', []) + [decision]
    
    return {
        'current_decision': decision,
        'decision_path': new_path
    }

def handle_urgent_case(state: DecisionTreeState):
    decision_path = state['decision_path']
    if decision_path[-1] == 'URGENT_CASE':
        # More detailed processing for urgent cases
        if 'server' in state['user_input'].lower() or 'down' in state['user_input'].lower():
            sub_decision = 'SERVER_OUTAGE'
        elif 'security' in state['user_input'].lower() or 'breach' in state['user_input'].lower():
            sub_decision = 'SECURITY_INCIDENT'
        else:
            sub_decision = 'HIGH_PRIORITY_GENERAL'
        
        new_path = state['decision_path'] + [sub_decision]
        
        return {
            'current_decision': sub_decision,
            'decision_path': new_path,
            'decision_result': f'Urgent case identified: {sub_decision}. Escalating immediately.'
        }
    return {'current_decision': 'URGENT_CASE'}

def handle_technical_issue(state: DecisionTreeState):
    decision_path = state['decision_path']
    if decision_path[-1] == 'TECHNICAL_ISSUE':
        # Determine type of technical issue
        user_input = state['user_input'].lower()
        
        if 'api' in user_input:
            sub_decision = 'API_ISSUE'
        elif 'ui' in user_input or 'interface' in user_input:
            sub_decision = 'UI_ISSUE'
        elif 'database' in user_input or 'db' in user_input:
            sub_decision = 'DATABASE_ISSUE'
        else:
            sub_decision = 'GENERAL_TECHNICAL_ISSUE'
        
        new_path = state['decision_path'] + [sub_decision]
        
        return {
            'current_decision': sub_decision,
            'decision_path': new_path,
            'decision_result': f'Technical issue identified: {sub_decision}. Routing to technical team.'
        }
    return {'current_decision': 'TECHNICAL_ISSUE'}

def handle_billing_question(state: DecisionTreeState):
    decision_path = state['decision_path']
    if decision_path[-1] == 'BILLING_QUESTION':
        user_input = state['user_input'].lower()
        
        if 'refund' in user_input:
            sub_decision = 'REFUND_REQUEST'
        elif 'subscription' in user_input:
            sub_decision = 'SUBSCRIPTION_QUERY'
        elif 'charge' in user_input:
            sub_decision = 'CHARGE_QUESTION'
        else:
            sub_decision = 'GENERAL_BILLING_QUERY'
        
        new_path = state['decision_path'] + [sub_decision]
        
        return {
            'current_decision': sub_decision,
            'decision_path': new_path,
            'decision_result': f'Billing question identified: {sub_decision}. Routing to billing department.'
        }
    return {'current_decision': 'BILLING_QUESTION'}

def handle_general_inquiry(state: DecisionTreeState):
    decision_path = state['decision_path']
    if decision_path[-1] == 'GENERAL_INQUIRY':
        user_input = state['user_input'].lower()
        
        if 'contact' in user_input or 'support' in user_input:
            sub_decision = 'CONTACT_INFORMATION'
        elif 'feature' in user_input or 'request' in user_input:
            sub_decision = 'FEATURE_REQUEST'
        else:
            sub_decision = 'GENERAL_INFORMATION'
        
        new_path = state['decision_path'] + [sub_decision]
        
        return {
            'current_decision': sub_decision,
            'decision_path': new_path,
            'decision_result': f'General inquiry identified: {sub_decision}. Providing standard response.'
        }
    return {'current_decision': 'GENERAL_INQUIRY'}

def route_based_on_decision(state: DecisionTreeState):
    current_decision = state['current_decision']
    
    if current_decision == 'URGENT_CASE':
        return 'handle_urgent_case'
    elif current_decision == 'TECHNICAL_ISSUE':
        return 'handle_technical_issue'
    elif current_decision == 'BILLING_QUESTION':
        return 'handle_billing_question'
    elif current_decision == 'GENERAL_INQUIRY':
        return 'handle_general_inquiry'
    else:
        return END

# Build the graph
def create_decision_tree_graph():
    workflow = StateGraph(DecisionTreeState)
    
    # Add nodes
    workflow.add_node('classify_input', classify_input)
    workflow.add_node('handle_urgent_case', handle_urgent_case)
    workflow.add_node('handle_technical_issue', handle_technical_issue)
    workflow.add_node('handle_billing_question', handle_billing_question)
    workflow.add_node('handle_general_inquiry', handle_general_inquiry)
    
    # Set entry point
    workflow.set_entry_point('classify_input')
    
    # Add conditional edges
    workflow.add_conditional_edges(
        'classify_input',
        route_based_on_decision,
        {
            'handle_urgent_case': 'handle_urgent_case',
            'handle_technical_issue': 'handle_technical_issue',
            'handle_billing_question': 'handle_billing_question',
            'handle_general_inquiry': 'handle_general_inquiry',
            END: END
        }
    )
    
    # Add edges for specialized handlers back to end
    workflow.add_edge('handle_urgent_case', END)
    workflow.add_edge('handle_technical_issue', END)
    workflow.add_edge('handle_billing_question', END)
    workflow.add_edge('handle_general_inquiry', END)
    
    return workflow.compile()

In [4]:
# Test the complex decision tree
tree_graph = create_decision_tree_graph()

# Example 1: Technical issue with API
initial_state = {
    'user_input': 'I am having issues with the API integration and getting authentication errors',
    'decision_path': [],
    'current_decision': '',
    'decision_result': '',
    'context': {}
}

result = tree_graph.invoke(initial_state)
print("Final state:", result)

Final state: {'user_input': 'I am having issues with the API integration and getting authentication errors', 'decision_path': ['TECHNICAL_ISSUE', 'API_ISSUE'], 'current_decision': 'API_ISSUE', 'decision_result': 'Technical issue identified: API_ISSUE. Routing to technical team.', 'context': {}}


## 2. Recursive Reasoning\n\nLearning objectives:\n- Implement loops and recursive thinking patterns\n- Handle iterative problem-solving approaches\n- Create self-referential reasoning systems

In [5]:
# Tutorial 2: Recursive Reasoning

# Define state for recursive reasoning
class RecursiveReasoningState(TypedDict):
    problem: str
    solution_steps: List[str]
    current_step: str
    iteration_count: int
    max_iterations: int
    final_solution: str
    needs_refinement: bool

# Decompose the problem into smaller parts
def decompose_problem(state: RecursiveReasoningState):
    problem = state['problem']
    iteration_count = state.get('iteration_count', 0)
    
    # For demonstration, we'll simulate breaking down a problem
    steps = []
    if iteration_count == 0:
        steps = [f"Step 1: Analyze problem '{problem}'"]
    elif iteration_count == 1:
        steps = [f"Step 2: Identify core components of '{problem}'"]
    elif iteration_count == 2:
        steps = [f"Step 3: Propose initial solution for '{problem}'"]
    else:
        steps = [f"Step {iteration_count + 1}: Refine solution based on feedback"]
    
    new_steps = state.get('solution_steps', []) + steps
    
    return {
        'solution_steps': new_steps,
        'current_step': steps[0],
        'iteration_count': iteration_count + 1
    }

# Evaluate the current solution
def evaluate_solution(state: RecursiveReasoningState):
    current_step = state['current_step']
    iteration_count = state['iteration_count']
    max_iterations = state.get('max_iterations', 5)
    
    # Simulate evaluation - in real scenarios this could involve external validation
    needs_refinement = iteration_count < 3  # Continue for 3 iterations
    
    # Generate a simulated evaluation
    evaluation = f"Evaluation of '{current_step}': Solution quality is {'medium' if needs_refinement else 'high'}"
    
    new_steps = state['solution_steps'] + [evaluation]
    
    return {
        'solution_steps': new_steps,
        'needs_refinement': needs_refinement,
        'current_step': evaluation
    }

# Refine the solution based on evaluation
def refine_solution(state: RecursiveReasoningState):
    problem = state['problem']
    iteration_count = state['iteration_count']
    
    refinement = f"Refinement {iteration_count}: Improving approach to '{problem}' based on previous evaluation"
    
    new_steps = state['solution_steps'] + [refinement]
    
    return {
        'solution_steps': new_steps,
        'current_step': refinement
    }

# Generate final solution when refinement is complete
def generate_final_solution(state: RecursiveReasoningState):
    solution_steps = state['solution_steps']
    problem = state['problem']
    
    final_solution = f"Final solution for '{problem}': After {len([s for s in solution_steps if 'Refinement' in s])} refinements, the comprehensive solution is ready."
    
    return {
        'final_solution': final_solution,
        'current_step': final_solution
    }

# Route based on whether refinement is needed
def should_refine(state: RecursiveReasoningState):
    needs_refinement = state.get('needs_refinement', True)
    iteration_count = state.get('iteration_count', 0)
    max_iterations = state.get('max_iterations', 5)
    
    if needs_refinement and iteration_count < max_iterations:
        return 'refine_solution'
    else:
        return 'generate_final_solution'

# Build the recursive reasoning graph
def create_recursive_reasoning_graph():
    workflow = StateGraph(RecursiveReasoningState)
    
    # Add nodes
    workflow.add_node('decompose_problem', decompose_problem)
    workflow.add_node('evaluate_solution', evaluate_solution)
    workflow.add_node('refine_solution', refine_solution)
    workflow.add_node('generate_final_solution', generate_final_solution)
    
    # Set entry point
    workflow.set_entry_point('decompose_problem')
    
    # Add edges
    workflow.add_edge('decompose_problem', 'evaluate_solution')
    workflow.add_conditional_edges(
        'evaluate_solution',
        should_refine,
        {
            'refine_solution': 'refine_solution',
            'generate_final_solution': 'generate_final_solution'
        }
    )
    workflow.add_edge('refine_solution', 'evaluate_solution')  # Loop back to evaluate
    workflow.add_edge('generate_final_solution', END)
    
    return workflow.compile()

In [6]:
# Test the recursive reasoning system
recursive_graph = create_recursive_reasoning_graph()

# Example: Problem solving through recursive reasoning
initial_state_recursive = {
    'problem': 'How to optimize database queries in our application?',
    'solution_steps': [],
    'current_step': '',
    'iteration_count': 0,
    'max_iterations': 5,
    'final_solution': '',
    'needs_refinement': True
}

result_recursive = recursive_graph.invoke(initial_state_recursive)
print("Final state:", result_recursive)

Final state: {'problem': 'How to optimize database queries in our application?', 'solution_steps': ["Step 1: Analyze problem 'How to optimize database queries in our application?'", "Evaluation of 'Step 1: Analyze problem 'How to optimize database queries in our application?'': Solution quality is medium", "Refinement 1: Improving approach to 'How to optimize database queries in our application?' based on previous evaluation", "Evaluation of 'Refinement 1: Improving approach to 'How to optimize database queries in our application?' based on previous evaluation': Solution quality is medium", "Refinement 2: Improving approach to 'How to optimize database queries in our application?' based on previous evaluation", "Evaluation of 'Refinement 2: Improving approach to 'How to optimize database queries in our application?' based on previous evaluation': Solution quality is high", "Final solution for 'How to optimize database queries in our application?': After 2 refinements, the comprehensive

## 3. Multi-Modal Graphs\n\nLearning objectives:\n- Work with different types of data (text, images, etc.)\n- Process multi-modal inputs in LangGraph\n- Implement cross-modal reasoning

In [7]:
# Tutorial 3: Multi-Modal Graphs

# For this example, we'll simulate multi-modal processing
# since we don't have actual image processing capabilities here

from typing import Union

# Define state for multi-modal processing
class MultiModalState(TypedDict):
    text_input: str
    image_descriptions: List[str]
    audio_transcripts: List[str]
    processed_content: List[str]
    cross_modal_analysis: str
    final_output: str
    modalities_present: List[str]

# Detect what modalities are present in the input
def detect_modalities(state: MultiModalState):
    text_input = state['text_input']
    
    modalities = ['text']
    
    # In a real implementation, this would detect actual image/audio inputs
    # For simulation, we'll check for keywords indicating different modalities
    if any(keyword in text_input.lower() for keyword in ['image:', 'picture:', 'photo:', 'visual:', 'see']):
        modalities.append('image')
    if any(keyword in text_input.lower() for keyword in ['audio:', 'sound:', 'voice:', 'hear', 'listen']):
        modalities.append('audio')
    
    return {
        'modalities_present': modalities
    }

# Process text modality
def process_text_modality(state: MultiModalState):
    text_input = state['text_input']
    
    # Extract meaningful information from text
    processed_text = f"Processed text: Extracted key concepts from '{text_input[:50]}...'"
    
    new_content = state.get('processed_content', []) + [processed_text]
    
    return {
        'processed_content': new_content
    }

# Process image modality (simulated)
def process_image_modality(state: MultiModalState):
    # In a real implementation, this would call an image processing model
    # For simulation, we'll generate a description based on text clues
    text_input = state['text_input']
    
    if 'product' in text_input.lower():
        image_desc = "Image of a product with high resolution and clear details"
    elif 'chart' in text_input.lower():
        image_desc = "Business chart showing sales metrics with clear visualization"
    else:
        image_desc = "General image with descriptive elements"
    
    new_images = state.get('image_descriptions', []) + [image_desc]
    processed_image = f"Processed image: {image_desc}"
    
    new_content = state.get('processed_content', []) + [processed_image]
    
    return {
        'image_descriptions': new_images,
        'processed_content': new_content
    }

# Process audio modality (simulated)
def process_audio_modality(state: MultiModalState):
    # In a real implementation, this would transcribe audio
    # For simulation, we'll generate a transcript based on text clues
    text_input = state['text_input']
    
    if 'meeting' in text_input.lower():
        transcript = "Meeting discussion about quarterly results and future strategy"
    elif 'interview' in text_input.lower():
        transcript = "Job interview covering experience and skills assessment"
    else:
        transcript = "General conversation with relevant content"
    
    new_audio = state.get('audio_transcripts', []) + [transcript]
    processed_audio = f"Processed audio: {transcript}"
    
    new_content = state.get('processed_content', []) + [processed_audio]
    
    return {
        'audio_transcripts': new_audio,
        'processed_content': new_content
    }

# Perform cross-modal analysis
def cross_modal_analysis(state: MultiModalState):
    text_input = state['text_input']
    image_descriptions = state.get('image_descriptions', [])
    audio_transcripts = state.get('audio_transcripts', [])
    
    # Combine insights from different modalities
    analysis_parts = [f"Analysis for query: '{text_input[:30]}...'"}
    
    if image_descriptions:
        analysis_parts.append(f"Images suggest: {image_descriptions[0][:50]}...")
    
    if audio_transcripts:
        analysis_parts.append(f"Audio indicates: {audio_transcripts[0][:50]}...")
    
    analysis = " | ".join(analysis_parts)
    
    return {
        'cross_modal_analysis': analysis
    }

# Generate final output combining all modalities
def generate_multimodal_output(state: MultiModalState):
    analysis = state['cross_modal_analysis']
    processed_content = state['processed_content']
    
    final_output = f"Multi-modal response: Based on analysis of all modalities, {analysis}. Key findings: {[c[:40] for c in processed_content[:2]]}"
    
    return {
        'final_output': final_output
    }

# Route to appropriate modality processors
def route_by_modality(state: MultiModalState):
    modalities = state.get('modalities_present', ['text'])
    
    routes = []
    if 'text' in modalities:
        routes.append('process_text_modality')
    if 'image' in modalities:
        routes.append('process_image_modality')
    if 'audio' in modalities:
        routes.append('process_audio_modality')
    
    # Return the first route, in a real implementation we'd handle multiple routes
    if routes:
        return routes[0]
    else:
        return 'cross_modal_analysis'

# Build the multi-modal graph
def create_multimodal_graph():
    workflow = StateGraph(MultiModalState)
    
    # Add nodes
    workflow.add_node('detect_modalities', detect_modalities)
    workflow.add_node('process_text_modality', process_text_modality)
    workflow.add_node('process_image_modality', process_image_modality)
    workflow.add_node('process_audio_modality', process_audio_modality)
    workflow.add_node('cross_modal_analysis', cross_modal_analysis)
    workflow.add_node('generate_multimodal_output', generate_multimodal_output)
    
    # Set entry point
    workflow.set_entry_point('detect_modalities')
    
    # Add edges
    workflow.add_edge('detect_modalities', 'process_text_modality')
    
    # Add conditional edge for other modalities
    def route_after_text(state: MultiModalState):
        modalities = state.get('modalities_present', [])
        if len(modalities) > 1:
            # If there are more modalities beyond text, process them
            if 'image' in modalities:
                return 'process_image_modality'
            elif 'audio' in modalities:
                return 'process_audio_modality'
        return 'cross_modal_analysis'
    
    workflow.add_conditional_edges(
        'process_text_modality',
        route_after_text,
        {
            'process_image_modality': 'process_image_modality',
            'process_audio_modality': 'process_audio_modality',
            'cross_modal_analysis': 'cross_modal_analysis'
        }
    )
    
    def route_after_image(state: MultiModalState):
        modalities = state.get('modalities_present', [])
        if 'audio' in modalities:
            return 'process_audio_modality'
        return 'cross_modal_analysis'
    
    workflow.add_conditional_edges(
        'process_image_modality',
        route_after_image,
        {
            'process_audio_modality': 'process_audio_modality',
            'cross_modal_analysis': 'cross_modal_analysis'
        }
    )
    
    workflow.add_edge('process_audio_modality', 'cross_modal_analysis')
    workflow.add_edge('cross_modal_analysis', 'generate_multimodal_output')
    workflow.add_edge('generate_multimodal_output', END)
    
    return workflow.compile()

In [8]:
# Test the multi-modal graph
multimodal_graph = create_multimodal_graph()

# Example: Input that suggests multiple modalities
initial_state_multimodal = {
    'text_input': 'Analyze this product image and the associated customer review audio',
    'image_descriptions': [],
    'audio_transcripts': [],
    'processed_content': [],
    'cross_modal_analysis': '',
    'final_output': '',
    'modalities_present': []
}

result_multimodal = multimodal_graph.invoke(initial_state_multimodal)
print("Final state:", result_multimodal)

Final state: {'text_input': 'Analyze this product image and the associated customer review audio', 'image_descriptions': ['Image of a product with high resolution and clear details'], 'audio_transcripts': ['General conversation with relevant content'], 'processed_content': ["Processed text: Extracted key concepts from 'Analyze this product image and the associat...'", 'Processed image: Image of a product with high resolution and clear details', 'Processed audio: General conversation with relevant content'], 'cross_modal_analysis': "Analysis for query: 'Analyze this product image and the associat...' | Images suggest: Image of a product with high resol... | Audio indicates: General conversation wit...", 'final_output': "Multi-modal response: Based on analysis of all modalities, Analysis for query: 'Analyze this product image and the associat...' | Images suggest: Image of a product with high resol... | Audio indicates: General conversation wit.... Key findings: ["Processed text: Extract

## 4. Performance Optimization\n\nLearning objectives:\n- Apply techniques for improving graph efficiency\n- Optimize resource utilization\n- Enhance execution speed and scalability

In [9]:
# Tutorial 4: Performance Optimization

import time
import asyncio
from concurrent.futures import ThreadPoolExecutor
import functools

# Define state for optimized processing
class OptimizedProcessingState(TypedDict):
    inputs: List[str]
    processed_results: List[str]
    processing_times: List[float]
    optimization_techniques_applied: List[str]
    performance_metrics: Dict[str, float]
    cache_hits: int
    total_processed: int

# Cache for storing previously computed results
processing_cache = {}

# Simulated expensive processing function
def expensive_processing(text: str) -> str:
    # Simulate some processing time
    time.sleep(0.1)
    return f"Processed: {text.upper()}"

# Async version of the processing function
async def async_expensive_processing(text: str) -> str:
    # Simulate some processing time
    await asyncio.sleep(0.1)
    return f"Async Processed: {text.upper()}"

# Parallel processing node
def parallel_process_inputs(state: OptimizedProcessingState):
    inputs = state['inputs']
    start_time = time.time()
    
    # Use ThreadPoolExecutor for parallel processing
    with ThreadPoolExecutor(max_workers=min(len(inputs), 4)) as executor:
        results = list(executor.map(expensive_processing, inputs))
    
    processing_time = time.time() - start_time
    
    # Calculate individual processing times (simulated)
    individual_times = [processing_time / len(inputs)] * len(inputs)
    
    new_results = state.get('processed_results', []) + results
    new_times = state.get('processing_times', []) + individual_times
    
    applied_optimizations = state.get('optimization_techniques_applied', []) + ['parallel_processing']
    
    return {
        'processed_results': new_results,
        'processing_times': new_times,
        'optimization_techniques_applied': applied_optimizations,
        'total_processed': len(new_results)
    }

# Caching node
def cache_enhanced_processing(state: OptimizedProcessingState):
    inputs = state['inputs']
    cache_hits = state.get('cache_hits', 0)
    results = []
    
    for inp in inputs:
        if inp in processing_cache:
            # Cache hit
            results.append(processing_cache[inp])
            cache_hits += 1
        else:
            # Cache miss - process and store in cache
            result = expensive_processing(inp)
            processing_cache[inp] = result
            results.append(result)
    
    new_results = state.get('processed_results', []) + results
    applied_optimizations = state.get('optimization_techniques_applied', []) + ['caching']
    
    return {
        'processed_results': new_results,
        'cache_hits': cache_hits,
        'optimization_techniques_applied': applied_optimizations,
        'total_processed': len(new_results)
    }

# Batch processing node
def batch_process_inputs(state: OptimizedProcessingState):
    inputs = state['inputs']
    batch_size = 3  # Small batch size for demonstration
    results = []
    times = []
    
    for i in range(0, len(inputs), batch_size):
        batch = inputs[i:i+batch_size]
        batch_start = time.time()
        
        # Process batch
        batch_results = [expensive_processing(item) for item in batch]
        batch_time = time.time() - batch_start
        
        results.extend(batch_results)
        times.extend([batch_time / len(batch)] * len(batch))
    
    new_results = state.get('processed_results', []) + results
    new_times = state.get('processing_times', []) + times
    applied_optimizations = state.get('optimization_techniques_applied', []) + ['batch_processing']
    
    return {
        'processed_results': new_results,
        'processing_times': new_times,
        'optimization_techniques_applied': applied_optimizations,
        'total_processed': len(new_results)
    }

# Asynchronous processing node
async def async_process_inputs(state: OptimizedProcessingState):
    inputs = state['inputs']
    start_time = time.time()
    
    # Run all async processes concurrently
    tasks = [async_expensive_processing(inp) for inp in inputs]
    results = await asyncio.gather(*tasks)
    
    processing_time = time.time() - start_time
    individual_times = [processing_time / len(inputs)] * len(inputs)
    
    new_results = state.get('processed_results', []) + results
    new_times = state.get('processing_times', []) + individual_times
    applied_optimizations = state.get('optimization_techniques_applied', []) + ['asynchronous_processing']
    
    return {
        'processed_results': new_results,
        'processing_times': new_times,
        'optimization_techniques_applied': applied_optimizations,
        'total_processed': len(new_results)
    }

# Calculate performance metrics
def calculate_performance_metrics(state: OptimizedProcessingState):
    processing_times = state.get('processing_times', [])
    total_processed = state.get('total_processed', 0)
    cache_hits = state.get('cache_hits', 0)
    
    if processing_times:
        avg_time = sum(processing_times) / len(processing_times)
        total_time = sum(processing_times)
    else:
        avg_time = 0
        total_time = 0
    
    metrics = {
        'average_processing_time': avg_time,
        'total_processing_time': total_time,
        'items_per_second': total_processed / total_time if total_time > 0 else 0,
        'cache_hit_rate': cache_hits / total_processed if total_processed > 0 else 0
    }
    
    return {
        'performance_metrics': metrics
    }

# Build the optimization graph
def create_optimization_graph():
    workflow = StateGraph(OptimizedProcessingState)
    
    # Add nodes
    workflow.add_node('parallel_process_inputs', parallel_process_inputs)
    workflow.add_node('cache_enhanced_processing', cache_enhanced_processing)
    workflow.add_node('batch_process_inputs', batch_process_inputs)
    # Note: We won't add the async node directly to the graph as it requires special handling
    workflow.add_node('calculate_performance_metrics', calculate_performance_metrics)
    
    # Set entry point - starting with parallel processing
    workflow.set_entry_point('parallel_process_inputs')
    
    # Add edges for optimization pipeline
    workflow.add_edge('parallel_process_inputs', 'cache_enhanced_processing')
    workflow.add_edge('cache_enhanced_processing', 'batch_process_inputs')
    workflow.add_edge('batch_process_inputs', 'calculate_performance_metrics')
    workflow.add_edge('calculate_performance_metrics', END)
    
    return workflow.compile()

# Function to demonstrate async processing separately
async def run_async_optimization(inputs: List[str]):
    start_time = time.time()
    results = await async_process_inputs({
        'inputs': inputs,
        'processed_results': [],
        'processing_times': [],
        'optimization_techniques_applied': [],
        'performance_metrics': {},
        'cache_hits': 0,
        'total_processed': 0
    })
    total_time = time.time() - start_time
    
    print(f"Async processing completed in {total_time:.2f}s for {len(inputs)} inputs")
    return results

In [10]:
# Test the optimization techniques
optimization_graph = create_optimization_graph()

# Example: Multiple inputs to process
initial_state_optimization = {
    'inputs': ['input1', 'input2', 'input3', 'input4'],
    'processed_results': [],
    'processing_times': [],
    'optimization_techniques_applied': [],
    'performance_metrics': {},
    'cache_hits': 0,
    'total_processed': 0
}

result_optimization = optimization_graph.invoke(initial_state_optimization)
print("Final state:", result_optimization)

# Demonstrate async processing
async_result = asyncio.run(run_async_optimization(['input1', 'input2', 'input3', 'input4']))

Final state: {'inputs': ['input1', 'input2', 'input3', 'input4'], 'processed_results': ['Processed: INPUT1', 'Processed: INPUT2', 'Processed: INPUT3', 'Processed: INPUT4', 'Processed: INPUT1', 'Processed: INPUT2', 'Processed: INPUT3', 'Processed: INPUT4', 'Processed: INPUT1', 'Processed: INPUT2', 'Processed: INPUT3', 'Processed: INPUT4'], 'processing_times': [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1], 'optimization_techniques_applied': ['parallel_processing', 'caching', 'batch_processing'], 'performance_metrics': {'average_processing_time': 0.1, 'total_processing_time': 1.2, 'items_per_second': 10.0, 'cache_hit_rate': 0.0}, 'cache_hits': 0, 'total_processed': 12}
Async processing completed in 0.11s for 4 inputs


## Summary

In this tutorial, we covered several advanced topics in LangGraph:

1. **Complex Decision Trees**: We built a sophisticated branching system that can classify and route inputs based on complex conditions.

2. **Recursive Reasoning**: We implemented a system that iteratively refines solutions through multiple rounds of evaluation and improvement.

3. **Multi-Modal Graphs**: We created a system that can process different types of inputs (text, image, audio) and combine insights across modalities.

4. **Performance Optimization**: We demonstrated various optimization techniques including parallel processing, caching, batch processing, and asynchronous execution.

These advanced techniques allow you to build more sophisticated and efficient LangGraph applications tailored to complex real-world requirements.